# Unequal Housing Conditions: EDA for Reactive Datasets

## Summary

In April 2022, Health Affairs journal [published research](https://drive.google.com/file/d/1UNsZFoiROnXif1XU6L2_kiELQ-xJmMia/view?usp=share_link) correlating racial makeup of neighborhoods to unequal housing conditions and code enforcement related to asthma triggers. The researchers found that, when analyzing publicly-available 311 data between 2011 and 2021, predominantly white neighborhoods report fewer conditions known to cause or exacerbate asthma, and when they do, they experience faster case closure rates, fewer overdue requests, and higher chances of the request resulting in a repair. 

While the above study was predicated on the publicly-available 311 data where renters call to put in their complaints before the City react (we term this data as reactive dataset), there has been a policy which allows the [City to randomly inspect housing](https://www.boston.gov/departments/inspectional-services/inspecting-housing-boston) to ensure safety and protect the health of the public.

Thus, the inspectional service department developed [an algorithm](https://drive.google.com/file/d/1Cn444cCKTDTU2sqXXCiqzDLha3wGvczA/view?usp=drive_link) to randomly assign houses for inspection. This proactive approach would be analyze to explore and determine the efficacy of the policy as it pertains to ensuring equitable distribution of the City's housing safety protocols. 


## EDA Problem Statement

Datasets: 
* 311 Requests, 2020 - Present
* Reactive Inspections data, 2023 - Present

Explore the different features in the dataset to see if it will good for feature selection and determining feature importance.
* If possible, a predictive model could be developed to using property features, location & other factors as IVs.
* Graph Analysis; to connect different locations using various nodes and edges to gain insights into likely violators.
* Similarity Analysis/CLustering based on different features.



In [1]:
#####  Import libraries for Boston City DoIT Analytics database..
import os 
import civis

# import libraries for data pre-processing
import pandas as pd
import numpy as np
from datetime import datetime
import datetime as datetime

# import libraries for visualization
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from chart_studio import plotly
import plotly.offline as pyoff
import plotly.graph_objs as go
import plotly.express as px


from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
init_notebook_mode(connected=True)        # Display plot on notebook while the kernel is running


# utils
import warnings 
warnings.filterwarnings('ignore')


In [2]:
os.environ['PYTHONPATH']='.'
os.environ['CIVIS_API_KEY']='iNtRtV8mWJYVuDKqr9XULieOXud8XndxiyWpFnSPUxs'

# Define the SQL query for reactive inspections/cases from 2023 - present 
sql_reactive_cases = """
SELECT * FROM isd_internal_data.reactive_cases
WHERE adddttm >= '2023-01-01 00:00:00+00'
"""
# Define the SQL query for reactive violations from 2023 - present 
sql_reactive_violations = """
SELECT * FROM isd_internal_data.reactive_violations 
WHERE adddttm >= '2023-01-01 00:00:00+00'
"""

df_reactive_inspections = civis.io.read_civis_sql(sql_reactive_cases,
                                            database='Boston',
                                            use_pandas=True)
df_reactive_violations = civis.io.read_civis_sql(sql_reactive_violations,
                                            database='Boston',
                                            use_pandas=True)


In [3]:
df_reactive_inspections.head()


,addby,adddttm,apby,apcasekey,apcaseprocessstatekey,apno,casegrp,caseinfostatus,casename,comments,...,empid,emplast,empmid,supr,suprflag,adddttm_fy,full_address,full_address_w_ward,typeno_translation_et,gps_geom_2249
0,172905,2023-05-25 12:31:59.667+00,172905,677067,1162,HVIOL-677067,GENERAL,Closed,101004851316,NaN,...,,NaN,NaN,,N,2022,First St,First St 06,NaN,0101000020C90800006B14C71C4DCD2741E7B0524DC37E...
1,140720,2023-01-24 15:51:19.327+00,140720,658030,1207,EVIOL-658030,,Closed,4641069,RODENT ACTIVITY,...,,NaN,NaN,,N,2022,99Dresser ST,99Dresser ST 06,NaN,0101000020C908000056F7958A0DC82741F480A2ED8B7E...
2,140720,2023-01-24 15:51:19.327+00,140720,658030,1207,EVIOL-658030,,Closed,4641069,RODENT ACTIVITY,...,,NaN,NaN,,N,2022,99Dresser ST,99Dresser ST 06,NaN,0101000020C908000056F7958A0DC82741F480A2ED8B7E...
3,140720,2023-01-24 15:51:19.327+00,140720,658030,1207,EVIOL-658030,,Closed,4641069,RODENT ACTIVITY,...,,NaN,NaN,,N,2022,99Dresser ST,99Dresser ST 06,NaN,0101000020C908000056F7958A0DC82741F480A2ED8B7E...
4,172905,2023-07-20 13:10:00.793+00,172905,686561,1160,HVIOL-686561,GENERAL,Open,NaN,NaN,...,,NaN,NaN,,N,2023,726Columbia RD,726Columbia RD 07,NaN,0101000020C908000083F04F35EEA62741687BD3F44572...


In [4]:
df_reactive_inspections.tail()


,addby,adddttm,apby,apcasekey,apcaseprocessstatekey,apno,casegrp,caseinfostatus,casename,comments,...,empid,emplast,empmid,supr,suprflag,adddttm_fy,full_address,full_address_w_ward,typeno_translation_et,gps_geom_2249
40001,162244,2023-08-08 18:13:40.763+00,162244,691918,1108,CE691918,,Open,NaN,NaN,...,162244,NaN,NaN,,N,2023,244Newbury ST,244Newbury ST 05,14:13:41,0101000020C9080000058878AB367827418504EF1FDB86...
40002,162244,2023-08-08 17:42:02.61+00,162244,691903,1108,CE691903,,Open,NaN,NaN,...,162244,NaN,NaN,,N,2023,73Newbury ST,73Newbury ST 05,13:42:04,0101000020C9080000007E0FAEF1882741F7FA4576C188...
40003,162244,2023-08-08 17:44:36.663+00,162244,691904,1108,CE691904,,Open,NaN,NaN,...,162244,NaN,NaN,,N,2023,85Newbury ST,85Newbury ST 05,13:44:37,0101000020C908000083781871DE872741B7465F6EA988...
40004,162244,2023-08-08 17:39:41.53+00,162244,691900,1108,CE691900,,Open,NaN,NaN,...,162244,NaN,NaN,,N,2023,71Newbury ST,71Newbury ST 05,13:39:42,0101000020C9080000FC7F23D12289274196344E70C488...
40005,135884,2023-06-20 13:22:31.35+00,135884,681154,1120,V681154,BLDG,Open,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2022,63Parker Hill AVE,63Parker Hill AVE 10,NaN,0101000020C90800007EF8E06F28402741644F69935D79...


In [5]:
df_reactive_violations.tail()


,addby,adddttm,apby,apcasekey,apcaseprocessstatekey,apno,casegrp,caseinfostatus,casename,comments,...,postdir,state,stname,stno,suffix,zip,adddttm_fy,full_address,full_address_w_ward,gps_geom_2249
48022,151799,2023-02-15 20:10:53.883+00,151799,661939,1144,CE661939,,Open,NaN,NaN,...,05,MA,Park,10,Pz,02116,2022,10Park Pz,10Park Pz 05,0101000020C9080000E8388EF37699274118C7719E6288...
48023,151799,2023-02-15 20:10:53.883+00,151799,661939,1144,CE661939,,Open,NaN,NaN,...,05,MA,Park,10,Pz,02116,2022,10Park Pz,10Park Pz 05,0101000020C9080000E8388EF37699274118C7719E6288...
48024,140965,2023-02-15 20:10:54.157+00,140965,661940,1104,CE661940,,Closed,NaN,NaN,...,09,MA,Shawmut,450,AVE,02116,2022,450Shawmut AVE,450Shawmut AVE 09,0101000020C90800008097BEEE1C882741A404B71A987F...
48025,74703,2023-02-15 20:19:35.31+00,074703,661941,1162,HVIOL-661941,GENERAL,Closed,101004663831,NaN,...,03,MA,Joy,37,ST,02114,2022,37Joy ST,37Joy ST 03,0101000020C90800008066CE85109D27418AC5B5242D8E...
48026,151799,2023-02-15 20:21:29.99+00,151799,661942,1144,CE661942,,Open,NaN,NaN,...,01,MA,Havre,227,ST,02128,2022,227Havre ST,227Havre ST 01,0101000020C9080000CB4BBCC2AADA2741C86A678C2D99...


In [6]:
sub = df_reactive_inspections[['apno','adddttm','scheddttm','compdttm','resultdttm','result',
                               'resultby','code','code_caseinspresult','gpsx','gpsy','gps_geom_2249']]


In [7]:
sub.head()

,apno,adddttm,scheddttm,compdttm,resultdttm,result,resultby,code,code_caseinspresult,gpsx,gpsy,gps_geom_2249
0,HVIOL-677067,2023-05-25 12:31:59.667+00,NaN,2023-06-20 19:20:00+00,2023-06-20 19:20:00+00,Served,106527,Closed,Served,779942.556206,2.948487e+06,0101000020C90800006B14C71C4DCD2741E7B0524DC37E...
1,EVIOL-658030,2023-01-24 15:51:19.327+00,NaN,NaN,NaN,,,Closed,,779270.770675,2.948376e+06,0101000020C908000056F7958A0DC82741F480A2ED8B7E...
2,EVIOL-658030,2023-01-24 15:51:19.327+00,NaN,NaN,NaN,,,Closed,,779270.770675,2.948376e+06,0101000020C908000056F7958A0DC82741F480A2ED8B7E...
3,EVIOL-658030,2023-01-24 15:51:19.327+00,NaN,NaN,NaN,,,Closed,,779270.770675,2.948376e+06,0101000020C908000056F7958A0DC82741F480A2ED8B7E...
4,HVIOL-686561,2023-07-20 13:10:00.793+00,NaN,NaN,NaN,,,Violation Inspection,,775031.104126,2.942092e+06,0101000020C908000083F04F35EEA62741687BD3F44572...


In [8]:
sub.tail()

,apno,adddttm,scheddttm,compdttm,resultdttm,result,resultby,code,code_caseinspresult,gpsx,gpsy,gps_geom_2249
40001,CE691918,2023-08-08 18:13:40.763+00,2023-08-08 18:13:41+00,2023-08-08 18:11:00+00,2023-08-08 18:11:00+00,Violation,162244,Fees,Violation,769051.334904,2.952630e+06,0101000020C9080000058878AB367827418504EF1FDB86...
40002,CE691903,2023-08-08 17:42:02.61+00,2023-08-08 17:42:04+00,2023-08-08 17:41:00+00,2023-08-08 17:41:00+00,Violation,162244,Fees,Violation,771192.839962,2.953603e+06,0101000020C9080000007E0FAEF1882741F7FA4576C188...
40003,CE691904,2023-08-08 17:44:36.663+00,2023-08-08 17:44:37+00,2023-08-08 17:43:00+00,2023-08-08 17:43:00+00,Violation,162244,Fees,Violation,771055.220890,2.953555e+06,0101000020C908000083781871DE872741B7465F6EA988...
40004,CE691900,2023-08-08 17:39:41.53+00,2023-08-08 17:39:42+00,2023-08-08 17:38:00+00,2023-08-08 17:38:00+00,Violation,162244,Fees,Violation,771217.408474,2.953609e+06,0101000020C9080000FC7F23D12289274196344E70C488...
40005,V681154,2023-06-20 13:22:31.35+00,NaN,NaN,NaN,NaN,NaN,Printing and Service,NaN,761876.218513,2.945723e+06,0101000020C90800007EF8E06F28402741644F69935D79...


### EDA

In [9]:
sub.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40006 entries, 0 to 40005
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   apno                 40006 non-null  object 
 1   adddttm              40006 non-null  object 
 2   scheddttm            37812 non-null  object 
 3   compdttm             37349 non-null  object 
 4   resultdttm           37822 non-null  object 
 5   result               39851 non-null  object 
 6   resultby             39851 non-null  object 
 7   code                 40006 non-null  object 
 8   code_caseinspresult  39851 non-null  object 
 9   gpsx                 39981 non-null  float64
 10  gpsy                 39981 non-null  float64
 11  gps_geom_2249        39981 non-null  object 
dtypes: float64(2), object(10)
memory usage: 3.7+ MB


* The missing records in the 'compdttm' means an inspection has not been completed in those units, explore its details..

In [10]:
# Count the number of empty rows in the 'compdttm' column.
empty_rows = sub['compdttm'].isnull().sum()

# Count the number of rows with empty strings in the 'compdttm' column.
empty_strings = (sub['compdttm'] == '').sum()

# Calculate the total number of empty rows or strings in the 'compdttm' column.
total_empty = empty_rows + empty_strings
total_empty


2657

#### Uninspected Units
**2657** reported cases were not inspected

In [11]:
# Drop rows where 'compdttm' column is either empty or contains an empty string.
sub.dropna(subset=['compdttm'], inplace=True)
sub.tail()


,apno,adddttm,scheddttm,compdttm,resultdttm,result,resultby,code,code_caseinspresult,gpsx,gpsy,gps_geom_2249
40000,CE690656,2023-08-02 00:39:39.98+00,2023-08-02 00:39:42+00,2023-08-02 00:38:00+00,2023-08-02 00:38:00+00,Violation,162244,Closed,Violation,770530.936217,2.948625e+06,0101000020C908000080E057DFC5832741E36BCF98087F...
40001,CE691918,2023-08-08 18:13:40.763+00,2023-08-08 18:13:41+00,2023-08-08 18:11:00+00,2023-08-08 18:11:00+00,Violation,162244,Fees,Violation,769051.334904,2.952630e+06,0101000020C9080000058878AB367827418504EF1FDB86...
40002,CE691903,2023-08-08 17:42:02.61+00,2023-08-08 17:42:04+00,2023-08-08 17:41:00+00,2023-08-08 17:41:00+00,Violation,162244,Fees,Violation,771192.839962,2.953603e+06,0101000020C9080000007E0FAEF1882741F7FA4576C188...
40003,CE691904,2023-08-08 17:44:36.663+00,2023-08-08 17:44:37+00,2023-08-08 17:43:00+00,2023-08-08 17:43:00+00,Violation,162244,Fees,Violation,771055.220890,2.953555e+06,0101000020C908000083781871DE872741B7465F6EA988...
40004,CE691900,2023-08-08 17:39:41.53+00,2023-08-08 17:39:42+00,2023-08-08 17:38:00+00,2023-08-08 17:38:00+00,Violation,162244,Fees,Violation,771217.408474,2.953609e+06,0101000020C9080000FC7F23D12289274196344E70C488...


In [12]:
# Look at the unique values from the 'result' column
unique_results = sub['result'].unique()
unique_results


array(['Served', 'NoViol', 'Violations', 'RP', 'OFA', 'Withdrawn',
       'OFANFC', 'NotServed', 'Violation', 'ServedM', 'CourtReins',
       'Corrected', 'UnitVacant', 'NollePros', 'Send Legal', 'Hearing',
       ' ', 'Failed', 'Condemn'], dtype=object)

In [13]:
# Look at the unique values from the 'code_caseinspresult' column
unique_code = sub['code_caseinspresult'].unique()
unique_code


array(['Served', 'NoViol', 'Violations', 'RP', 'OFA', 'Withdrawn',
       'OFANFC', 'NotServed', 'Violation', 'ServedM', 'CourtReins',
       'Corrected', 'UnitVacant', 'NollePros', 'Send Legal', 'Hearing',
       ' ', 'Failed', 'Condemn'], dtype=object)

### How to Categorized Code Result?
* Should we have three categories:
    1. Fail: Violation(s), Failed, Condemn, RP, CourtReins, 
    2. Pass: NoViol, Withdrawn,  Corrected, 
    3. IC: 'OFA', 'OFANFC', UnitVacant,**
* What about (CourtReins, NollePros, RP, OFA, etc.
* Drop:  Served, ServedM

In [15]:
# Create new categories for Pass, Fail and IC.
sub['result'] = sub['result'].replace({'NoViol': 'Pass', 'Withdrawn': 'Pass', 'Corrected': 'Pass'})
sub['result'] = sub['result'].replace({'Failed': 'Fail', 'Violation': 'Fail', 'Violations': 'Fail',
                                       'Condemn': 'Fail', 'RP': 'Fail', 'CourtReins': 'Fail'})
sub['result'] = sub['result'].replace({'OFA': 'IC', 'OFANFC': 'IC', 'UnitVacant': 'IC'})


In [16]:
# List of valid result categories
valid_results = ['Pass', 'Fail', 'IC']

# Filter the DataFrame to keep only valid result categories
sub = sub[sub['result'].isin(valid_results)]
unique_results = sub['result'].unique()
unique_results


array(['Pass', 'Fail', 'IC'], dtype=object)

In [17]:
# Convert 'Fail' to 0 and 'Pass' to 1 and 'IC' to 2
sub['result'] = sub['result'].map({'Fail': 0, 'Pass': 1, 'IC': 2})

#Print the unique values in the 'result' column to verify the changes
unique_results = sub['result'].unique()
unique_results


array([1, 0, 2], dtype=int64)

In [18]:
# Calculate the count and percentage of the results

result_counts = sub['result'].value_counts()
result_percentages = sub['result'].value_counts(normalize=True) * 100

# Create a DataFrame to display the result
result_table = pd.DataFrame({'Count': result_counts, 'Percentage': result_percentages})

# Format the percentage column to display with two decimal places
result_table['Percentage'] = result_table['Percentage'].map('{:.2f}%'.format)

# Display the result table
print(result_table)


        Count Percentage
result                  
0       33984     94.18%
2        1280      3.55%
1         820      2.27%


For reactive case, 94% failed inspections, which means renters are mostly right, with only 2.27% passing inspections and 3.55& on incomplete.

#### Time Duration Analysis
* For case duration: Using 'adddttm' for the date the inspection was determined and 'resultdttm' for when the inspection was resulted.
**resdttm**

In [22]:
# Drop rows with missing values in the 'scheddttm' column
sub = sub.dropna(subset=['scheddttm'])
sub = sub.dropna(subset=['compdttm'])
sub = sub.dropna(subset=['scheddttm'])
sub = sub.dropna(subset=['resultdttm'])


In [24]:
from dateutil import parser

# Convert 'adddtttm', 'scheddttm', 'compdttm' and 'resultdttm' columns to datetime format without seconds and milliseconds.
sub['adddttm'] = sub['adddttm'].apply(lambda x: parser.parse(x).strftime('%Y-%m-%d %H:%M'))
sub['scheddttm'] = sub['scheddttm'].apply(lambda x: parser.parse(x).strftime('%Y-%m-%d %H:%M'))
sub['compdttm'] = sub['compdttm'].apply(lambda x: parser.parse(x).strftime('%Y-%m-%d %H:%M'))
sub['resultdttm'] = sub['resultdttm'].apply(lambda x: parser.parse(x).strftime('%Y-%m-%d %H:%M'))

# Calculate the time difference between when the case was reported 'adddttm', scheduled for inspection, completed and resulted.
sub['case_to_sched_time'] = pd.to_datetime(sub['scheddttm'], format='%Y-%m-%d %H:%M') - pd.to_datetime(sub['adddttm'], format='%Y-%m-%d %H:%M')
sub['case_to_comp_time'] = pd.to_datetime(sub['compdttm'], format='%Y-%m-%d %H:%M') - pd.to_datetime(sub['adddttm'], format='%Y-%m-%d %H:%M')
sub['case_to_result_time'] = pd.to_datetime(sub['resultdttm'], format='%Y-%m-%d %H:%M') - pd.to_datetime(sub['adddttm'], format='%Y-%m-%d %H:%M')
sub['sched_to_comp_time'] = pd.to_datetime(sub['compdttm'], format='%Y-%m-%d %H:%M') - pd.to_datetime(sub['scheddttm'], format='%Y-%m-%d %H:%M')
sub['sched_to_result_time'] = pd.to_datetime(sub['resultdttm'], format='%Y-%m-%d %H:%M') - pd.to_datetime(sub['scheddttm'], format='%Y-%m-%d %H:%M')

# Convert the time difference to days and hours for case reported and schedule date time
sub['case_to_sched_days'] = sub['case_to_sched_time'] / pd.Timedelta(days=1)
sub['case_to_sched_hours'] = sub['case_to_sched_time'] / pd.Timedelta(hours=1)

# Convert the time difference to days and hours for case reported and inspection completed date time
sub['case_to_comp_days'] = sub['case_to_comp_time'] / pd.Timedelta(days=1)
sub['case_to_comp_hours'] = sub['case_to_comp_time'] / pd.Timedelta(hours=1)

# Convert the time difference to days and hours for case reported and inspection resulted date time
sub['case_to_result_days'] = sub['case_to_result_time'] / pd.Timedelta(days=1)
sub['case_to_result_hours'] = sub['case_to_result_time'] / pd.Timedelta(hours=1)

# Convert the time difference to days and hours for case inspection scheduled and completed date time
sub['sched_to_comp_days'] = sub['sched_to_comp_time'] / pd.Timedelta(days=1)
sub['sched_to_comp_hours'] = sub['sched_to_comp_time'] / pd.Timedelta(hours=1)

# Convert the time difference to days and hours for case inspection scheduled and resulted date time
sub['sched_to_result_time'] = sub['sched_to_result_time'] / pd.Timedelta(days=1)
# sub['sched_to_result_time'] = sub['sched_to_result_time'] / pd.Timedelta(hours=1)


In [25]:
sub.head()


,apno,adddttm,scheddttm,compdttm,resultdttm,result,resultby,code,code_caseinspresult,gpsx,...,sched_to_comp_time,sched_to_result_time,case_to_sched_days,case_to_sched_hours,case_to_comp_days,case_to_comp_hours,case_to_result_days,case_to_result_hours,sched_to_comp_days,sched_to_comp_hours
26,HVIOL-676622,2023-05-23 13:07,2023-12-25 20:11,2023-07-19 19:29,2023-07-19 19:29,0,139460,Legal,RP,782996.267173,...,-160 days +23:18:00,-159.029167,216.294444,5191.066667,57.265278,1374.366667,57.265278,1374.366667,-159.029167,-3816.700000
169,HVIOL-668955,2023-03-30 14:09,2023-05-17 04:00,2023-05-22 04:00,2023-05-23 12:18,2,106527,Closed,OFA,782176.928004,...,5 days 00:00:00,6.345833,47.577083,1141.850000,52.577083,1261.850000,53.922917,1294.150000,5.000000,120.000000
338,HVIOL-674071,2023-05-03 13:14,2023-05-09 14:00,2023-05-09 15:00,2023-05-10 12:04,2,050088,Closed,OFANFC,763388.042195,...,0 days 01:00:00,0.919444,6.031944,144.766667,6.073611,145.766667,6.951389,166.833333,0.041667,1.000000
339,HVIOL-666091,2023-03-13 17:21,2023-05-09 13:28,2023-05-09 17:50,2023-05-10 12:32,2,050088,Closed,OFA,745748.509346,...,0 days 04:22:00,0.961111,56.838194,1364.116667,57.020139,1368.483333,57.799306,1387.183333,0.181944,4.366667
340,HVIOL-667772,2023-03-22 19:27,2023-03-24 13:00,2023-03-24 15:55,2023-03-27 12:35,2,050088,Closed,OFANFC,770759.085562,...,0 days 02:55:00,2.982639,1.731250,41.550000,1.852778,44.466667,4.713889,113.133333,0.121528,2.916667
